In [1]:
from pprint import pprint
import datasets as datasets_lib
import grain
import pandas as pd
import os
import fsspec
import numpy as np

import transformers
from tunix.generate import mappings

Dataset = datasets_lib.Dataset
AutoTokenizer = transformers.AutoTokenizer

from absl import logging as absl_logging

import logging
import sys

# ====== Logging Configuration ======
# 1. Force absl to use python logging
absl_logging.use_python_logging()

# 2. Configure the root logger
logging.basicConfig(
    stream=sys.stdout,
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - [%(name)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    force=True,
)

# 3. Explicitly set levels for relevant loggers
logging.getLogger().setLevel(logging.INFO)
logging.getLogger("absl").setLevel(logging.INFO)

# 4. Set absl verbosity
absl_logging.set_verbosity(absl_logging.INFO)
absl_logging.set_stderrthreshold("info")

print("Logging configured at INFO level.")

/home/linchai_google_com/miniconda3/envs/tunix/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Logging configured at INFO level.


In [ ]:
try:
  from GOOGLE_INTERNAL_PACKAGE_PATH.pyglib import gfile
  from etils import ecolab

  cm = ecolab.adhoc(
      source=ecolab.FROM_NOTEBOOK_OR_HEAD,
      reload="tunix",
      behavior="preferred",
      cell_autoreload=True,
  )

  file_open = gfile.Open

  NOTEBOOK_ENV = "g3"
except Exception:
  NOTEBOOK_ENV = "git"

  import contextlib
  cm = contextlib.nullcontext()

  file_open = fsspec.open


In [3]:

with cm:
  from tunix.models.qwen2 import model as qwen2_lib
  from tunix.models.qwen2 import params as qwen2_params_lib
  from tunix.models.gemma4 import model as gemma4_lib
  from tunix.models.gemma4 import params_safetensors as gemma4_params_lib
  from tunix.generate import sampler as sampler_lib
  from tunix.utils import math_utils
# %%
from typing import Any, Dict, Optional
import jax
from jax import numpy as jnp
from flax import nnx
import orbax.checkpoint as ocp
from tqdm.auto import tqdm
import re

In [4]:

MODEL_PATH_PREFIX = "gs://tunix/models"
MODEL_MAPPING = {
    "Qwen/Qwen2.5-1.5B-Instruct": (
        qwen2_lib.ModelConfig.qwen2p5_1p5b(),
        os.path.join(MODEL_PATH_PREFIX, "qwen2_5/torch/1.5b-it"),
    ),
    "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B": (
        qwen2_lib.ModelConfig.deepseek_r1_distill_qwen_1p5b(),
        os.path.join(MODEL_PATH_PREFIX, "DeepSeek-R1-Distill-Qwen-1.5B"),
    ),
    "agentica-org/DeepScaleR-1.5B-Preview": (
        qwen2_lib.ModelConfig.deepseek_r1_distill_qwen_1p5b(),
        os.path.join(MODEL_PATH_PREFIX, "DeepScaleR-1.5B-Preview"),
    ),
    "google/gemma-4-31B-it": (
      gemma4_lib.ModelConfig.gemma4_31b(),
      "/mnt/disks/linchai-data/huggingface/hub/models--google--gemma-4-31B-it/snapshots/439edf5652646a0d1bd8b46bfdc1d3645761a445",
    ),
    "google/gemma-4-E2B-it": (
      gemma4_lib.ModelConfig.gemma4_e2b(),
      "/mnt/disks/linchai-data/huggingface/hub/models--google--gemma-4-E2B-it/snapshots/b4a601102c3d45e2b7b50e2057a6d5ec8ed4adcf",
    ),
    "google/gemma-4-E4B-it": (
      gemma4_lib.ModelConfig.gemma4_e4b(),
      "/mnt/disks/linchai-data/huggingface/hub/models--google--gemma-4-E4B-it/snapshots/83df0a889143b1dbfc61b591bbc639540fd9ce4c",
    ),
    "google/gemma-4-26B-A4B-it": (
      gemma4_lib.ModelConfig.gemma4_26b_a4b(),
      "/mnt/disks/linchai-data/huggingface/hub/models--google--gemma-4-26B-A4B-it/snapshots/7d4c97e54145f8ffd1a4dd1b4986a5015a517842",
    ),
    
}

In [5]:
import os
os.environ["TPU_LOG_DIR"] = "~/my_tpu_logs"
os.environ["SKIP_JAX_PRECOMPILE"] = "True"

In [6]:

# model_version = "google/gemma-4-26B-A4B-it"
model_version = "google/gemma-4-31B-it"
model_config, model_path = MODEL_MAPPING[model_version]

tokenizer = AutoTokenizer.from_pretrained(model_version)

2026-05-06 04:23:51 - INFO - [httpx] HTTP Request: HEAD https://huggingface.co/google/gemma-4-31B-it/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"


2026-05-06 04:23:51 - WARNING - [huggingface_hub.utils._http] Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-05-06 04:23:51 - INFO - [httpx] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/gemma-4-31B-it/145dc2508c480a64b47242f160d286cff94a2343/config.json "HTTP/1.1 200 OK"


2026-05-06 04:23:51 - INFO - [httpx] HTTP Request: HEAD https://huggingface.co/google/gemma-4-31B-it/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-06 04:23:51 - INFO - [httpx] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/gemma-4-31B-it/145dc2508c480a64b47242f160d286cff94a2343/tokenizer_config.json "HTTP/1.1 200 OK"
2026-05-06 04:23:51 - INFO - [httpx] HTTP Request: GET https://huggingface.co/api/models/google/gemma-4-31B-it/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-05-06 04:23:51 - INFO - [httpx] HTTP Request: GET https://huggingface.co/api/models/google/gemma-4-31B-it/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-05-06 04:23:54 - INFO - [httpx] HTTP Request: GET https://huggingface.co/api/models/google/gemma-4-31B-it "HTTP/1.1 200 OK"


In [7]:

trainer_mesh = jax.sharding.Mesh(np.array(jax.devices()).reshape(1, 4), axis_names=["fsdp", "tp"])
rollout_mesh = jax.sharding.Mesh(np.array(jax.devices()).reshape(1, 4), axis_names=["fsdp", "tp"])


Could not access ~/my_tpu_logs: not found


In [8]:

print("Loading model from safe tensors...")
with trainer_mesh:
  trainer_model = gemma4_params_lib.create_model_from_safe_tensors(file_dir=model_path, config=model_config, mesh=trainer_mesh)

import jax
jax.clear_caches()

Loading model from safe tensors...
2026-05-06 04:24:10 - WARNING - [absl] Skipped loading 356 keys because they could not be mapped to model weights. This may be expected, for example when loading only text weights from a multimodal checkpoint. Missing keys: [model.embed_vision.embedding_projection.weight, model.vision_tower.encoder.layers.0.input_layernorm.weight, model.vision_tower.encoder.layers.0.mlp.down_proj.linear.weight, model.vision_tower.encoder.layers.0.mlp.gate_proj.linear.weight, model.vision_tower.encoder.layers.0.mlp.up_proj.linear.weight, model.vision_tower.encoder.layers.0.post_attention_layernorm.weight, model.vision_tower.encoder.layers.0.post_feedforward_layernorm.weight, model.vision_tower.encoder.layers.0.pre_feedforward_layernorm.weight, model.vision_tower.encoder.layers.0.self_attn.k_norm.weight, model.vision_tower.encoder.layers.0.self_attn.k_proj.linear.weight, model.vision_tower.encoder.layers.0.self_attn.o_proj.linear.weight, model.vision_tower.encoder.layer

In [8]:
from tunix.generate import vllm_sampler   # pylint: disable=g-import-not-at-top

# mapping_config = mappings.MappingConfig.build(
#     mapping_obj=None,
#     model=trainer_model,
#     backend="vllm_jax",
# )
mapping_config = mappings.MappingConfig()

INFO 05-06 04:03:59 [__init__.py:59] TPU info: node_name=maxtext-single-host-1-v5p-8 | tpu_type=v5p-8 | worker_id=0 | num_chips=4 | num_cores_per_chip=2
2026-05-06 04:03:59 - WARNING - [absl] Tensorflow library not found, tensorflow.io.gfile operations will use native shim calls. GCS paths (i.e. 'gs://...') cannot be accessed.


/home/linchai_google_com/miniconda3/envs/tunix/lib/python3.12/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()
/home/linchai_google_com/miniconda3/envs/tunix/lib/python3.12/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()


In [10]:
sampler_vllm = vllm_sampler.VllmSampler(
    tokenizer=tokenizer,
    config=vllm_sampler.VllmConfig(
        mesh=rollout_mesh,
        hbm_utilization=0.78,
        enable_dp_attention=True,
        init_with_random_weights=False,
        mapping_config=mappings.MappingConfig(),
        engine_kwargs={
            "model": model_version,
            "max_model_len": 64,
            "max_num_seqs": 1,
            "max_num_batched_tokens": 2496,
        },
        return_logprobs=True,
    ),
)

2026-05-06 04:07:41 - INFO - [absl] Engine kwargs setting key 'model' with value 'google/gemma-4-31B-it'.
2026-05-06 04:07:41 - INFO - [absl] Engine kwargs setting key 'max_model_len' with value '64'.
2026-05-06 04:07:41 - INFO - [absl] Engine kwargs setting key 'max_num_seqs' with value '1'.
2026-05-06 04:07:41 - INFO - [absl] Engine kwargs setting key 'max_num_batched_tokens' with value '2496'.
INFO 05-06 04:07:41 [utils.py:233] non-default args: {'max_model_len': 64, 'tensor_parallel_size': 4, 'gpu_memory_utilization': 0.78, 'max_num_batched_tokens': 2496, 'max_num_seqs': 1, 'disable_log_stats': True, 'additional_config': {'sharding': {'sharding_strategy': {'expert_parallelism': 1, 'device_indexes': [0, 1, 2, 3], 'enable_dp_attention': True}}}, 'model': 'google/gemma-4-31B-it'}
WARNING 05-06 04:07:41 [arg_utils.py:1491] The global random seed is set to 0. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.


Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  50% Completed | 1/2 [03:21<03:21, 201.80s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [04:01<00:00, 106.58s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [04:01<00:00, 120.86s/it]



INFO 05-06 04:12:13 [default_loader.py:391] Loading weights took 242.09 seconds


W0506 04:12:15.463895 2316955 pjrt_executable.cc:642] Assume version compatibility. PjRt-IFRT does not track XLA executable versions.


INFO 05-06 04:12:16 [tpu_runner.py:616] Init model | hbm=[(14.8, 95.74), (14.8, 95.74), (14.8, 95.74), (14.8, 95.74)]GiB
INFO 05-06 04:12:16 [tpu_worker.py:380] Memory statistics | total_hbm_limit_gb=382.97GiB | total_hbm_limit_cap_gb=298.72GiB | total_hbm_used_gb=59.22GiB | total_hbm_avail_gb=239.5GiB
INFO 05-06 04:12:16 [kv_cache_utils.py:1710] GPU KV cache size: 285,376 tokens
INFO 05-06 04:12:16 [kv_cache_utils.py:1711] Maximum concurrency for 64 tokens per request: 4459.00x
INFO 05-06 04:12:21 [kv_cache_manager.py:834] Hybrid KV cache layout: num_kv_cache_groups=1, num_kv_cache_tensors=60, kv_cache_config.num_blocks=17836, duplicate_shared_layers=False
INFO 05-06 04:12:21 [kv_cache_manager.py:862] Init kv-cache | num_total_layers=60 | num_blocks=[17836, 17836, 17836, 17836, 17836, 17836, 17836, 17836, 17836, 17836, 17836, 17836, 17836, 17836, 17836, 17836, 17836, 17836, 17836, 17836, 17836, 17836, 17836, 17836, 17836, 17836, 17836, 17836, 17836, 17836, 17836, 17836, 17836, 17836, 

In [11]:
jax.clear_caches()
prompts = [{"role": "user", "content": "which is bigger, 1 or 2"}]
prompts = tokenizer.apply_chat_template(prompts, add_generation_prompt=True, tokenize=False)
out_data = sampler_vllm(
    input_strings=prompts,
    max_generation_steps=16,
    max_prompt_length=32,
    temperature=0,
    top_p=1,
    top_k=None,
    seed=None,
    echo=False,
    pad_output=False,
)
print("Text from VLLM sampler: ", out_data.text)
print("logprobs from VLLM sampler: ", out_data.logprobs)
print("tokens from VLLM sampler: ", out_data.tokens)
print("padded_prompt_tokens from VLLM sampler: ", out_data.padded_prompt_tokens)

WARNING 05-06 04:12:42 [model.py:1449] Default vLLM sampling parameters have been overridden by the model's `generation_config.json`: `{'temperature': 1.0, 'top_k': 64, 'top_p': 0.95}`. If this is not intended, please relaunch vLLM instance with `--generation-config vllm`.
type of input_strings:  <class 'list'>
type of each input string:  <class 'str'>


Processed prompts: 100%|██████████| 1/1 [00:38<00:00, 38.58s/it, est. speed input: 0.57 toks/s, output: 0.21 toks/s]

Text from VLLM sampler:  ['2 is bigger than 1.<turn|>']
logprobs from VLLM sampler:  [[-1.8954058759845793e-05, -3.337797534186393e-05, 0.0, -0.0013262758729979396, 0.0, 0.0, -1.1920901954454166e-07, -4.7683607817816664e-07]]
tokens from VLLM sampler:  [array([236778,    563,  12869,   1082, 236743, 236770, 236761,    106],
      dtype=int32)]
padded_prompt_tokens from VLLM sampler:  [[     0      0      0      0      0      0      0      0      0      0
       2    105   2364    107   7650    563  12869 236764 236743 236770
     653 236743 236778    106    107    105   4368    107    100  45518
     107    101]]


In [9]:

out_tokens = np.array([236778,    563,  12869,   1082, 236743, 236770, 236761,    106], dtype=np.int32)


prompt_tokens = np.array([     0 ,     0,      0,      0,      0,      0,      0,      0,      0,      0,
       2,    105,   2364,    107,   7650,    563,  12869, 236764, 236743, 236770,
     653, 236743, 236778,    106,    107,    105,   4368,    107,    100,  45518,
     107,    101], dtype=np.int32)
completion_tokens = out_tokens
print(f"{prompt_tokens=}")
print(f"{completion_tokens=}")
prompt_tokens = jnp.repeat(jnp.array([prompt_tokens]), 1, axis=0)
completion_tokens = jnp.repeat(jnp.array([completion_tokens]), 1, axis=0)


from tunix.rl import common
graphdef, state = nnx.split(trainer_model)

jax.clear_caches()

ref_logps = common.compute_per_token_logps(
    graphdef,
    state,
    prompt_tokens=prompt_tokens,
    completion_tokens=completion_tokens,
    pad_id=tokenizer.pad_token_id,
    eos_id=tokenizer.eos_token_id,
    temperature = 0,
)

prompt_tokens=array([     0,      0,      0,      0,      0,      0,      0,      0,
            0,      0,      2,    105,   2364,    107,   7650,    563,
        12869, 236764, 236743, 236770,    653, 236743, 236778,    106,
          107,    105,   4368,    107,    100,  45518,    107,    101],
      dtype=int32)
completion_tokens=array([236778,    563,  12869,   1082, 236743, 236770, 236761,    106],
      dtype=int32)


prompt_completion_ids = Array([[     0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      2,    105,   2364,    107,   7650,    563,
         12869, 236764, 236743, 236770,    653, 236743, 236778,    106,
           107,    105,   4368,    107,    100,  45518,    107,    101,
        236778,    563,  12869,   1082, 236743, 236770, 236761,    106]],      dtype=int32)
prompt_mask = Array([[False, False, False, False, False, False, False, False, False,
        False,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True]], dtype=bool)
prompt_completion_mask = Array([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]],      dtype=int32)
input_tokens = Array([[     0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      2,    105,   23

In [ ]:
import jax
jax.clear_caches()
jax.block_until_ready(ref_logps)
print(f"{ref_logps[0] = }")

# float32 [-1.6212287e-05, -4.2914307e-05,  0.0000000e+00, -1.2116224e-03,  0.0000000e+00,  0.0000000e+00, -1.1920902e-07, -5.9604508e-07]
# bf16 [-1.6212287e-05, -4.1364681e-05,  0.0000000e+00, -1.2746054e-03, 0.0000000e+00,  0.0000000e+00, -1.1920902e-07, -4.7683608e-07]
# vllm sampler: [-1.8954058759845793e-05, -3.337797534186393e-05, 0.0, -0.0013262758729979396, 0.0, 0.0, -1.1920901954454166e-07, -4.7683607817816664e-07]

ref_logps[0] = Array([-1.6212287e-05, -4.1364681e-05,  0.0000000e+00, -1.2746054e-03,
        0.0000000e+00,  0.0000000e+00, -1.1920902e-07, -4.7683608e-07],      dtype=float32)


: 